In [ ]:
import os
import zipfile
import numpy as np
from PIL import Image
import shutil
from google.colab import files

zip_input = "/content/Dataset.zip"
extract_path = "/content/dataset_raw"
output_dir = "/content/dataset_preprocessed"
TARGET_SIZE = (224, 224)

if os.path.exists(extract_path): shutil.rmtree(extract_path)
if os.path.exists(output_dir): shutil.rmtree(output_dir)
os.makedirs(output_dir)

# ekstrak
print(f"ekstrak file {zip_input}")
with zipfile.ZipFile(zip_input, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

all_folders = [f for f in os.listdir(extract_path) if os.path.isdir(os.path.join(extract_path, f)) and not f.startswith('__')]
base_path = extract_path
if len(all_folders) == 1:
    base_path = os.path.join(extract_path, all_folders[0])

print("mulai preprocessing")

# pemrosesan normalisasi
array_normalisasi = None

for class_name in os.listdir(base_path):
    class_path = os.path.join(base_path, class_name)
    if not os.path.isdir(class_path): continue

    output_class_path = os.path.join(output_dir, class_name)
    os.makedirs(output_class_path, exist_ok=True)

    print(f"sedang memproses kelas: {class_name}")
    for img_name in os.listdir(class_path):
        try:
            img_path = os.path.join(class_path, img_name)
            img = Image.open(img_path)

            # rgp dan resize
            img_rgb = img.convert('RGB')
            img_resized = img_rgb.resize(TARGET_SIZE)

            # simpan jadi jpg
            new_name = os.path.splitext(img_name)[0] + ".jpg"
            img_resized.save(os.path.join(output_class_path, new_name), "JPEG", quality=95)

            # normalisasi untuk output teks
            if array_normalisasi is None:
                array_normalisasi = np.array(img_resized) / 255.0
        except:
            continue

print("\n preprocessing selesai")

# donwload file
zip_output = "/content/dataset_preprocessing"
print("\n jadikan ke zip")
shutil.make_archive(zip_output, 'zip', output_dir)

print(f"status normalisasi: sukses (pixel {array_normalisasi.min():.2f} - {array_normalisasi.max():.2f})")
print(f"ukuran gambar: {TARGET_SIZE[0]}x{TARGET_SIZE[1]} pixel")

print("\n download file")
files.download(zip_output + ".zip")